# Setup

In [1]:
import bw2io as bi
import bw2data as bd
import bw2calc as bc
import git

import networkx as nx
from networkx.algorithms import bipartite
import random
from collections import defaultdict
from matplotlib import pyplot as plt 
import json
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.patches as mpatches
import geopandas as gpd
import re
from shapely.geometry import Point
import pickle
import gzip
import scipy

from datetime import datetime
from functools import wraps
import time


In [2]:
def get_git_root():
    repo = git.Repo(search_parent_directories=True)
    return repo.git.rev_parse("--show-toplevel")
root = get_git_root()

In [3]:

def get_current_time():
    return datetime.now().strftime("%H:%M:%S")


In [4]:
bd.projects.set_current('SEE_LAB') 
eidb = bd.Database("ecoinvent-3.9.1-cutoff")
ei_biosphere = bd.Database("ecoinvent-3.9.1-biosphere")
wb = bd.Database("Water bottle LCA")


# Exploration

In [5]:
with open(f"{root}/Data/saved_networks/eco_3.9.1/eco_3-9-1_graph", 'rb') as f:
          G = pickle.load(f)

In [18]:
def get_random_node(graph):
    a = random.choice(list(graph.nodes))
    sphere = graph.nodes[a]['sphere']
    if sphere == 'biosphere':
        return get_random_node(graph=graph)
    else:
        return a

In [22]:
def print_dict(dictionary):
    print(json.dumps(dictionary, indent=4))

In [6]:
len(list(G.nodes))

25958

In [7]:
sccs = list(nx.strongly_connected_components(G))
subgraphs = [G.subgraph(scc).copy() for scc in sccs if len(scc) > 1]
adj_matrices = [
    nx.to_numpy_array(subgraph, nodelist=sorted(subgraph.nodes()))
    for subgraph in subgraphs]

In [8]:
print(adj_matrices[1])

[[1. 0. 0. 1. 0. 0.]
 [0. 1. 1. 0. 0. 0.]
 [1. 0. 1. 0. 1. 0.]
 [0. 1. 0. 1. 0. 1.]
 [0. 0. 0. 1. 1. 0.]
 [0. 0. 1. 0. 0. 1.]]


In [9]:
print(len(adj_matrices))

16


In [19]:
G1 = subgraphs[-1]
get_random_node(G1)

'7934a31b85c3f48438213b9b6654a017'

In [89]:
print_dict(G1.nodes[get_random_node(G1)])

{
    "comment": "Rough estimation.\n[This dataset was already contained in the ecoinvent database version 2. It was not individually updated during the transfer to ecoinvent version 3. Life Cycle Impact Assessment results may still have changed, as they are affected by changes in the supply chain, i.e. in other datasets. This dataset was generated following the ecoinvent quality guidelines for version 2. It may have been subject to central changes described in the ecoinvent version 3 change report (http://www.ecoinvent.org/database/ecoinvent-version-3/reports-of-changes/), and the results of the central updates were reviewed extensively. The changes added e.g. consistent water flows and other information throughout the database. The documentation of this dataset can be found in the ecoinvent reports of version 2, which are still available via the ecoinvent website. The change report linked above covers all central changes that were made during the conversion process.]\nIncluded activi

In [ ]:
for i, mat in enumerate(adj_matrices):
    print(i, mat.shape)

0 (6, 6)
1 (6, 6)
2 (13, 13)
3 (10, 10)
4 (6, 6)
5 (2, 2)
6 (6, 6)
7 (6, 6)
8 (3, 3)
9 (6, 6)
10 (6, 6)
11 (4, 4)
12 (6, 6)
13 (6, 6)
14 (5, 5)
15 (13709, 13709)


In [11]:
# eigenvalues = []
# for mat in adj_matrices:
#     start_time = get_current_time()
#     eig_vals = scipy.linalg.eigvals(mat)
#     end_time = get_current_time()
#     eigenvalues.append(eig_vals)
#     print(f"Eigenvalues computed at {start_time}. Time: {end_time}")